# SciFact Qwen3 generation (Colab T4)

Use a free Colab T4 runtime. Recommended model: `Qwen/Qwen3-4B`; fallback: `Qwen/Qwen3-1.7B`. Both are public Hugging Face repositories, so no Hugging Face token is required. Set `QWEN_MODEL_ID` to one of these IDs (or another compatible Qwen3 model) when prompted. The notebook only uses the uploaded prompt export and never connects to the project database.

Export `generation_prompts.jsonl` and `generation_prompts.meta.json` from the repository first. Download the two generated output files at the end, then import them using the documented command. Segment files stay on the Colab VM and allow resuming after a runtime disconnect if the VM disk survives.

In [ ]:
%pip install -q transformers==4.51.3 accelerate==1.6.0 huggingface_hub==0.30.2 safetensors==0.5.3

In Colab, select **Runtime → Change runtime type → T4 GPU** before continuing. Install torch from the Colab image to keep its CUDA build compatible.

In [ ]:
from google.colab import files
uploaded = files.upload()
required_uploads = {"generation_prompts.jsonl", "generation_prompts.meta.json"}
if not required_uploads.issubset(uploaded):
    raise ValueError(f"Upload both files: {sorted(required_uploads)}")

In [ ]:
import hashlib
import json
from pathlib import Path

PROMPTS_PATH = Path("generation_prompts.jsonl")
SIDECAR_PATH = Path("generation_prompts.meta.json")
prompt_bytes = PROMPTS_PATH.read_bytes()
prompt_sidecar = json.loads(SIDECAR_PATH.read_text(encoding="utf-8"))
prompt_sha256 = hashlib.sha256(prompt_bytes).hexdigest()
if prompt_sidecar.get("sha256") != prompt_sha256:
    raise ValueError("Prompt checksum does not match the sidecar")
with PROMPTS_PATH.open(encoding="utf-8") as prompt_file:
    prompts = [json.loads(line) for line in prompt_file if line.strip()]
if len(prompts) != prompt_sidecar.get("row_count") or len(prompts) != 600:
    raise ValueError(f"Expected 600 prompt rows; found {len(prompts)}")
if any(row.get("prompt_version") != prompt_sidecar.get("prompt_version") for row in prompts):
    raise ValueError("Prompt version does not match the sidecar")
prompt_keys = [(row["key"]["claim_id"], row["key"]["retriever"]) for row in prompts]
if len(set(prompt_keys)) != len(prompts):
    raise ValueError("Prompt export contains duplicate keys")
print(f"Verified {len(prompts)} prompts; SHA-256 {prompt_sha256}; version {prompt_sidecar['prompt_version']}")

In [ ]:
import os
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is available. Select a Colab T4 GPU runtime.")
device_name = torch.cuda.get_device_name(0)
if "T4" not in device_name:
    print(f"Warning: expected a T4 GPU, found {device_name}")
if torch.cuda.is_bf16_supported():
    print("This GPU supports bfloat16; this workflow still uses fp16 for T4 compatibility.")
print(f"GPU in use: {device_name}; CUDA {torch.version.cuda}; fp16")

QWEN_MODEL_ID = os.environ.get("QWEN_MODEL_ID") or input("QWEN_MODEL_ID (for example Qwen/Qwen3-4B): ").strip()
if not QWEN_MODEL_ID:
    raise ValueError("QWEN_MODEL_ID is required; no model default is selected by the notebook")
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
)
model.eval()
revision_hash = getattr(model.config, "_commit_hash", None) or tokenizer.init_kwargs.get("_commit_hash")
if not revision_hash:
    raise RuntimeError("Hugging Face did not report the resolved repository commit hash")
print(f"Loaded {QWEN_MODEL_ID} at revision {revision_hash}; dtype={next(model.parameters()).dtype}")

BATCH_SIZE = 2
MAX_NEW_TOKENS = 256
SEGMENT_SIZE = 50
SEGMENTS_DIR = Path("generation_segments")
SEGMENTS_DIR.mkdir(exist_ok=True)
segment_manifest_path = SEGMENTS_DIR / "manifest.json"
segment_manifest = {"prompts_sha256": prompt_sha256, "model_id": QWEN_MODEL_ID, "revision_hash": revision_hash}
if segment_manifest_path.exists():
    if json.loads(segment_manifest_path.read_text(encoding="utf-8")) != segment_manifest:
        raise ValueError("Existing generation segments belong to a different prompt file or model revision")
else:
    segment_manifest_path.write_text(json.dumps(segment_manifest, indent=2) + "\n", encoding="utf-8")
print("Batch size 2 balances T4 throughput and memory; max_new_tokens=256 provides room for the JSON object, citations, and a rationale bounded to 60 words.")

In [ ]:
def chat_text(prompt_text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

eos_ids = model.generation_config.eos_token_id
if eos_ids is None:
    eos_ids = tokenizer.eos_token_id
if isinstance(eos_ids, int):
    eos_ids = {eos_ids}
else:
    eos_ids = set(eos_ids)

def generate_batch(items):
    messages = [chat_text(item["prompt"]) for item in items]
    encoded = tokenizer(messages, return_tensors="pt", padding=True, truncation=False)
    encoded = {name: tensor.to("cuda") for name, tensor in encoded.items()}
    with torch.inference_mode():
        sequences = model.generate(
            **encoded,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = sequences[:, encoded["input_ids"].shape[1]:]
    results = []
    for item, token_ids in zip(items, generated):
        ids = token_ids.tolist()
        stop_at = next((index for index, token_id in enumerate(ids) if token_id in eos_ids), None)
        emitted_ids = ids[:stop_at + 1] if stop_at is not None else ids
        text = tokenizer.decode(emitted_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        results.append({
            "key": item["key"],
            "raw_output_text": text,
            "new_tokens": len(emitted_ids),
            "hit_max_new_tokens": stop_at is None and len(emitted_ids) == MAX_NEW_TOKENS,
        })
    return results

first_ten = prompts[:10]
first_pass = [result for start in range(0, len(first_ten), BATCH_SIZE) for result in generate_batch(first_ten[start:start + BATCH_SIZE])]
second_pass = [result for start in range(0, len(first_ten), BATCH_SIZE) for result in generate_batch(first_ten[start:start + BATCH_SIZE])]
determinism_check_passed = all(a["raw_output_text"] == b["raw_output_text"] for a, b in zip(first_pass, second_pass))
if not determinism_check_passed:
    for index, (first, second) in enumerate(zip(first_pass, second_pass)):
        if first["raw_output_text"] != second["raw_output_text"]:
            print(f"Determinism mismatch at prompt {index}:\nFIRST: {first['raw_output_text']}\nSECOND: {second['raw_output_text']}")
    raise RuntimeError("Greedy determinism check failed; stop before the full run")
print("Determinism check passed: 10 prompts generated twice with identical raw output.")
for index in range(3):
    print(f"\n--- PROMPT {index + 1} ---\n{first_ten[index]['prompt']}\n--- RAW OUTPUT {index + 1} ---\n{first_pass[index]['raw_output_text']}")

In [ ]:
import time

elapsed_start = time.perf_counter()
for segment_start in range(0, len(prompts), SEGMENT_SIZE):
    segment_end = min(segment_start + SEGMENT_SIZE, len(prompts))
    segment_path = SEGMENTS_DIR / f"segment_{segment_start:04d}_{segment_end:04d}.jsonl"
    expected = prompts[segment_start:segment_end]
    expected_keys = [(row["key"]["claim_id"], row["key"]["retriever"]) for row in expected]
    if segment_path.exists():
        saved = [json.loads(line) for line in segment_path.read_text(encoding="utf-8").splitlines() if line]
        saved_keys = [(row["key"]["claim_id"], row["key"]["retriever"]) for row in saved]
        if saved_keys != expected_keys:
            raise ValueError(f"Existing segment {segment_path} does not match this prompt export")
        print(f"Resuming: kept segment {segment_start}-{segment_end}")
        continue
    segment_rows = []
    for batch_start in range(segment_start, segment_end, BATCH_SIZE):
        batch = prompts[batch_start:min(batch_start + BATCH_SIZE, segment_end)]
        segment_rows.extend(generate_batch(batch))
    temporary_path = segment_path.with_suffix(".tmp")
    temporary_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in segment_rows), encoding="utf-8")
    temporary_path.replace(segment_path)
    print(f"Saved segment {segment_start}-{segment_end}")

all_results = []
for segment_start in range(0, len(prompts), SEGMENT_SIZE):
    segment_end = min(segment_start + SEGMENT_SIZE, len(prompts))
    segment_path = SEGMENTS_DIR / f"segment_{segment_start:04d}_{segment_end:04d}.jsonl"
    all_results.extend(json.loads(line) for line in segment_path.read_text(encoding="utf-8").splitlines() if line)
actual_keys = [(row["key"]["claim_id"], row["key"]["retriever"]) for row in all_results]
if actual_keys != prompt_keys or len(set(actual_keys)) != len(prompts):
    raise ValueError("Merged generation keys do not match the prompt export exactly")
OUTPUT_PATH = Path("generations.jsonl")
OUTPUT_PATH.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in all_results), encoding="utf-8")
elapsed_seconds = time.perf_counter() - elapsed_start
import importlib.metadata
generations_meta = {
    "model_id": QWEN_MODEL_ID,
    "revision_hash": revision_hash,
    "dtype": str(next(model.parameters()).dtype),
    "library_versions": {name: importlib.metadata.version(name) for name in ("torch", "transformers", "accelerate", "huggingface_hub", "safetensors")},
    "batch_size": BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "device_name": device_name,
    "elapsed_seconds": elapsed_seconds,
    "determinism_check_passed": determinism_check_passed,
    "think_output_count": sum("<think>" in row["raw_output_text"] for row in all_results),
    "prompt_version": prompt_sidecar["prompt_version"],
    "prompts_sha256": prompt_sha256,
    "output_count": len(all_results),
}
Path("generations_meta.json").write_text(json.dumps(generations_meta, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Generated {len(all_results)} rows in {elapsed_seconds:.1f} seconds; {generations_meta['think_output_count']} contain <think>.")
print("Review generations_meta.json and the segment files before downloading.")

In [ ]:
files.download("generations.jsonl")
files.download("generations_meta.json")